# 📊 Model Evaluation — WER, BLEU, ROUGE
**JSS SMC MCA Institute | Udaya Kumar | USN: P02ME24S126024**


In [ ]:
!pip install -q transformers datasets jiwer sacrebleu rouge-score evaluate
from google.colab import drive
drive.mount('/content/drive')
print('✅ Ready!')

In [ ]:
# ── 1. Whisper ASR — Word Error Rate (WER) ──
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from datasets import load_from_disk
from jiwer import wer as compute_wer
import torch

WHISPER_PATH = '/content/drive/MyDrive/audio_project/models/whisper-kn-hi'
processor = WhisperProcessor.from_pretrained(WHISPER_PATH)
model     = WhisperForConditionalGeneration.from_pretrained(WHISPER_PATH)
model.eval()

test_data  = load_from_disk('/content/drive/MyDrive/audio_project/data/kn_processed')

# Use first 200 samples for speed
NUM_SAMPLES = 200
refs, hyps  = [], []

for sample in test_data.select(range(NUM_SAMPLES)):
    audio   = sample['audio']['array']
    inputs  = processor(audio, sampling_rate=16000, return_tensors='pt')
    with torch.no_grad():
        ids = model.generate(inputs.input_features)
    pred = processor.batch_decode(ids, skip_special_tokens=True)[0]
    refs.append(sample['sentence'])
    hyps.append(pred)

wer_score = compute_wer(refs, hyps)
print(f'\n📊 ASR Word Error Rate (WER): {wer_score*100:.2f}%')
print(f'   Target: < 35% WER')
print(f'   Status: {"✅ PASS" if wer_score < 0.35 else "❌ Needs more training"}')

In [ ]:
# ── 2. Translation — BLEU Score ──
import sacrebleu

# Replace these with your actual model outputs and references
hypotheses = [
    'Today we learn about data structures.',
    'An algorithm is a sequence of steps to solve a problem.',
    'The stack is a last in first out data structure.',
]
references = [
    ['Today we will learn about data structures.'],
    ['An algorithm is a set of instructions to solve a problem.'],
    ['Stack is a Last-In-First-Out data structure.'],
]

bleu = sacrebleu.corpus_bleu(hypotheses, references)
print(f'\n📊 Translation BLEU Score: {bleu.score:.2f}')
print(f'   Target: > 20 BLEU')
print(f'   Status: {"✅ PASS" if bleu.score > 20 else "❌ Below target"}')

In [ ]:
# ── 3. Note Structuring — ROUGE Score ──
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# Replace with actual model-generated notes vs human-written reference
generated = '''
KEY POINTS:
1. Data structures organize data efficiently for algorithms.
2. Stacks use LIFO ordering; queues use FIFO.
3. Binary search enables O(log n) search in sorted arrays.
'''

reference = '''
Main topics:
Data structures are fundamental to organizing information.
Stack (LIFO) and Queue (FIFO) are linear data structures.
Binary search is efficient for finding elements in sorted sequences.
'''

scores = scorer.score(reference, generated)
print(f'\n📊 Note Structuring ROUGE Scores:')
print(f'   ROUGE-1:  {scores["rouge1"].fmeasure:.3f}')
print(f'   ROUGE-2:  {scores["rouge2"].fmeasure:.3f}')
print(f'   ROUGE-L:  {scores["rougeL"].fmeasure:.3f}')
print(f'   Target:   > 0.30 ROUGE-L')
print(f'   Status:   {"✅ PASS" if scores["rougeL"].fmeasure > 0.30 else "❌ Below target"}')

In [ ]:
# ── 4. Language Detection Accuracy ──
from langdetect import detect
from sklearn.metrics import accuracy_score, classification_report

# Sample test data: (text, expected_language)
test_samples = [
    ('Today we learn about algorithms.', 'en'),
    ('Binary search uses divide and conquer.', 'en'),
    ('ಈ ಪಾಠದಲ್ಲಿ ನಾವು ಡೇಟಾ ರಚನೆಗಳ ಬಗ್ಗೆ ಕಲಿಯುತ್ತೇವೆ', 'kn'),
    ('ಅಲ್ಗಾರಿದಮ್ ಎಂದರೆ ಸಮಸ್ಯೆ ಪರಿಹರಿಸುವ ಹಂತಗಳ ಸಮೂಹ', 'kn'),
    ('आज हम डेटा स्ट्रक्चर के बारे में पढ़ेंगे', 'hi'),
    ('स्टैक एक LIFO डेटा संरचना है', 'hi'),
]

y_true, y_pred = [], []
for text, expected in test_samples:
    try:
        predicted = detect(text)
        # Normalize: only track kn/hi/en
        predicted = predicted if predicted in ['en','kn','hi'] else 'unknown'
    except:
        predicted = 'unknown'
    y_true.append(expected)
    y_pred.append(predicted)

acc = accuracy_score(y_true, y_pred)
print(f'\n📊 Language Detection Accuracy: {acc*100:.1f}%')
print(f'   Target: > 85%')
print(f'   Status: {"✅ PASS" if acc > 0.85 else "❌ Below target"}')
print('\nDetailed Report:')
print(classification_report(y_true, y_pred))

In [ ]:
# ── 5. Final Results Table ──
print('=' * 60)
print('   FINAL EVALUATION RESULTS')
print('   JSS SMC MCA Institute | Udaya Kumar')
print('=' * 60)
print(f'{"Module":<25} {"Metric":<15} {"Score":<12} {"Target"}')
print('-' * 60)
print(f'{"Whisper ASR":<25} {"WER":<15} {wer_score*100:.1f}%{"":<8} < 35%')
print(f'{"IndicTrans2":<25} {"BLEU":<15} {bleu.score:.1f}{"":<10} > 20')
print(f'{"T5 Notes":<25} {"ROUGE-L":<15} {scores["rougeL"].fmeasure:.3f}{"":<8} > 0.30')
print(f'{"Lang Detection":<25} {"Accuracy":<15} {acc*100:.1f}%{"":<8} > 85%')
print('=' * 60)